In [ ]:
from agents import Agent, Runner, function_tool, ItemHelpers

@function_tool
def get_weather(city: str):
    """Get the weather in a specific city"""
    print(city)
    return "30 degress"

agent = Agent(
    name="Assistant Agent",
    instructions="You ar a helpful Assistant. Use tools when needed to answer questions",
    tools=[get_weather],
)

stream = Runner.run_streamed(agent, "Hello how are you? what is the weather in tokyo?")

async for event in stream.stream_events():
    if event.type == "raw_response_event":
        continue
    elif event.type == "agent_updated_stream_event":
        print("Agent updated to", event.new_agent.name)
    elif event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            print(event.item.raw_item.to_dict())
        elif event.item.type == "tool_call_output_item":
            print(event.item.output)
        elif event.item.type == "message_output_item":
            print(ItemHelpers.text_message_output(event.item))
    print("=" * 20)


In [ ]:
from agents import Agent, Runner, function_tool, SQLiteSession

session = SQLiteSession("user_1", "ai-memory.db")

@function_tool
def get_weather(city: str):
    """Get the weather in a specific city"""
    print(city)
    return "30 degress"

agent = Agent(
    name="Assistant Agent",
    instructions="You ar a helpful Assistant. Use tools when needed to answer questions",
    tools=[get_weather],
)

result = await Runner.run(
    agent,
    "私は日本に住んでいる。",
    session=session,
)

print(result.final_output)


In [ ]:
result = await Runner.run(
    agent,
    "私が住んでいる国の中で首都の明日の天気は？",
    session=session,
)

print(result.final_output)


In [ ]:
# 履歴が空のセッションに手動で文脈を注入する例
session2 = SQLiteSession("user_2", "ai-memory.db")

await session2.add_items([{"role": "user", "content": "私は日本に住んでいる。"}])

result = await Runner.run(
    agent,
    "私が住んでいる国の中で首都の明日の天気は？",
    session=session2,
)

print(result.final_output)
